In [2]:
import numpy as np

n = 10
matrix = np.zeros((n, n))
matrix_f = np.zeros(n)
for i in range(1, n+1):
    for j in range(1, n+1):
        if i == j:
            matrix[i-1][j-1] = 1
        else:
            matrix[i-1][j-1] = 1/(i+j)
for i in range(1, n+1):
    matrix_f[i-1] = 1/i

print(matrix_f)

[1.         0.5        0.33333333 0.25       0.2        0.16666667
 0.14285714 0.125      0.11111111 0.1       ]


In [3]:
def gauss_meth(a, b):
    n = len(b)
    a_loc = np.copy(a)
    result = np.copy(b)

    forward_iterations = 0
    backward_iterations = 0

    for k in range(0, n-1):
        for i in range(k+1, n):
            if a_loc[i, k] != 0.0:
                lam = a_loc[i, k] / a_loc[k, k]
                a_loc[i, k+1:n] = a_loc[i, k+1:n] - lam * a_loc[k, k+1:n]
                result[i] = result[i] - lam * result[k]
                forward_iterations += 1

    for k in range(n-1, -1, -1):
        result[k] = (result[k] - np.dot(a_loc[k, k+1:n], result[k+1:n])) / a_loc[k, k]
        backward_iterations += 1

    print(f'Количество итераций в прямом ходе: {forward_iterations}')
    print(f'Количество итераций в обратном ходе: {backward_iterations}')

    return result

def norma_matrix(a):
    s = 0
    for i in range(len(a)):
        for j in range(len(a[0])):   
            s += a[i,j]**2
    result = s**0.5
    return result 

def norma_vector(a):
    s = 0
    for i in range(len(a)):
        s += a[i]**2
    result = s**0.5
    return result

def matr_difference(a, b):
        if a.shape == b.shape:
            return a - b
        else:
            print('Ошибка: матрицы разных размеров')
            return None
    
def invert_matrix(a):
    a_loc = np.copy(a)
    result = np.eye(len(a_loc),len(a_loc))
    for fd in range(len(a_loc)):
        fd_scaler = 1.0 / a_loc[fd][fd]
        for j in range(len(a_loc)):
            a_loc[fd][j] *= fd_scaler
            result[fd][j] *= fd_scaler
        
        for i in list(range(len(a_loc)))[0:fd] + list(range(len(a_loc)))[fd+1:]:
            cr_scaler = a_loc[i][fd]
            
            for j in range(len(a_loc)):
                a_loc[i][j] = a_loc[i][j] - cr_scaler * a_loc[fd][j]
                result[i][j] = result[i][j] - cr_scaler * result[fd][j]
    return result

def sor(a, b, w, x0, tol):
    
    n = len(b)
    if x0 is None:
        x0 = np.zeros(n)
    
    x = np.copy(x0)
    iterations = 0
    
    while True:
        x_old = np.copy(x)
        
        for i in range(n):

            sigma = np.dot(a[i, :i], x[:i]) + np.dot(a[i, i+1:], x_old[i+1:])
            x[i] = (1 - w) * x_old[i] + (w / a[i, i]) * (b[i] - sigma)

        iterations += 1
            
        if np.linalg.norm(x - x_old, ord=np.inf) < tol:
            print('Достигнута сходимость')
            return x, iterations

In [4]:
x1 = gauss_meth(matrix, matrix_f)
print(x1)

Количество итераций в прямом ходе: 45
Количество итераций в обратном ходе: 10
[ 9.19077109e-01  1.75540170e-01  6.39348240e-02  2.72747640e-02
  1.14234685e-02  3.51083928e-03 -7.89957814e-04 -3.25080145e-03
 -4.69787781e-03 -5.55373994e-03]


In [5]:
lam = np.linalg.eig(matrix)[0]
max_lam = max(lam)
min_lam = min(lam)
print('Максимальная лямбда:', max(lam))
print('Минимальная лямбда:', min(lam))
print(lam)

matrix_inv = invert_matrix(matrix)

print('Число обусловленности матрицы:', norma_matrix(matrix)*norma_matrix(matrix_inv))

Максимальная лямбда: 2.0483599269774446
Минимальная лямбда: 0.6579597538101806
[2.04835993 0.65795975 0.81224305 0.86881409 0.97922572 0.89874464
 0.91743388 0.93029794 0.94715386 0.93976714]
Число обусловленности матрицы: 11.768936014445325


In [6]:
r1 = matr_difference(np.dot(matrix, x1), matrix_f)
print('Норма вектора невязки: ', norma_vector(r1))

Норма вектора невязки:  3.925231146709438e-17


In [7]:
x2, iters = sor(matrix, matrix_f, 1.5 ,x0 = None, tol = 0.000001)
print(x2)
print('Количество итераций: ', iters)

Достигнута сходимость
[ 9.19077139e-01  1.75539951e-01  6.39346588e-02  2.72746612e-02
  1.14234125e-02  3.51081603e-03 -7.89958418e-04 -3.25078654e-03
 -4.69785245e-03 -5.55370774e-03]
Количество итераций:  30


In [8]:
r2 = matr_difference(np.dot(matrix, x2), matrix_f)
print('Норма вектора невязки: ', norma_vector(r2))

Норма вектора невязки:  4.2228871538320777e-07


In [9]:
def power_iteration(A, tol):
    n, _ = A.shape
    b_k = np.random.rand(n)
    iters = 0
    while True:
        b_k1 = np.dot(A, b_k)
        b_k1_norm = np.linalg.norm(b_k1)
        b_k1 = b_k1 / b_k1_norm
        iters += 1
        
        if np.linalg.norm(b_k1 - b_k) < tol:
            break
        
        b_k = b_k1

    value = np.dot(b_k.T, np.dot(A, b_k))
    print(iters)
    return value, b_k


def inverse_iteration(A, shift, tol):
    n, _ = A.shape
    b_k = np.random.rand(n)
    
    M = A - shift * np.eye(n)
    
    iters = 0
    
    while True:
        b_k1 = np.linalg.solve(M, b_k)
        b_k1_norm = np.linalg.norm(b_k1)
        b_k1 = b_k1 / b_k1_norm
        iters += 1
        
        if np.linalg.norm(b_k1 - b_k) < tol:
            break
        
        b_k = b_k1

    value = np.dot(b_k.T, np.dot(A, b_k))
    print(iters)
    return value, b_k


In [26]:
max_value, max_vector = power_iteration(matrix, 0.001)
print("Максимальное собственное значение:", max_value)
print("Соответствующий собственный вектор:", max_vector)

shift = 0.65
min_value, min_vector = inverse_iteration(matrix, shift, 0.001)
print("Минимальное собственное значение:", min_value)
print("Соответствующий собственный вектор:", min_vector)

print("Разница для максимума", abs(max_value-max_lam))
print("Разница для минимума",abs(min_value-min_lam))

9
Максимальное собственное значение: 2.0483585574076533
Соответствующий собственный вектор: [0.46195687 0.42507624 0.37430211 0.33224223 0.29729817 0.26972828
 0.24576741 0.22680601 0.20979084 0.19602231]
4
Минимальное собственное значение: 0.6579597542056914
Соответствующий собственный вектор: [-0.77735276  0.60113713  0.16445823  0.07390024  0.03732672  0.01887689
  0.00844487  0.00212066 -0.00189328 -0.00447961]
Разница для максимума 1.3695697913540528e-06
Разница для минимума 3.955108462960766e-10
